In [1]:
import os
import zipfile
import subprocess
import time
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from google.cloud import storage
from google.api_core import exceptions
import shutil

# 1. 环境初始化
PROJECT_ID = "gen-lang-client-0371685655"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)

MODEL_NAME = "gemini-2.0-flash" 
model = GenerativeModel(MODEL_NAME)

# 2. 路径配置 (已修改关键词及对应目录)
BUCKET_NAME = 'xhs-humor-data'
SEARCH_KEYWORD = "脱口秀集锦"  # 修改此处
LOCAL_BASE_DIR = './talkshow_collection_content' # 同步修改目录名
CLOUD_RESULT_PREFIX = 'talkshow_collection_content' # 同步修改云端前缀

os.makedirs(LOCAL_BASE_DIR, exist_ok=True)
RESULT_BASE_DIR = LOCAL_BASE_DIR 

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)

# --- [断点续传]：预加载已完成列表 ---
print(f"正在同步云端 [{CLOUD_RESULT_PREFIX}] 下已完成的任务列表...")
existing_blobs = set([b.name for b in bucket.list_blobs(prefix=CLOUD_RESULT_PREFIX)])
print(f"云端已存在 {len(existing_blobs)} 个解析结果，这些将被跳过。\n")

# 3. 获取目标文件并预扫描总量
all_blobs = storage_client.list_blobs(BUCKET_NAME)
target_zips = [b for b in all_blobs if SEARCH_KEYWORD in b.name and b.name.endswith('.zip')]

print(f"正在预扫描视频总量以计算全局进度...")
total_videos = 0
task_list = []

for zip_blob in target_zips:
    temp_zip = os.path.join(LOCAL_BASE_DIR, "scan.zip")
    zip_blob.download_to_filename(temp_zip)
    with zipfile.ZipFile(temp_zip, 'r') as z:
        v_list = [f for f in z.namelist() if f.lower().endswith(('.mp4', '.mov', '.avi')) and not f.startswith('__MACOSX')]
        total_videos += len(v_list)
        task_list.append((zip_blob, v_list))
    os.remove(temp_zip)

print(f"共计待处理视频总数: {total_videos}\n")

# --- 定义带重试机制的请求函数 ---
def generate_with_retry(prompt_text, audio_bytes, max_retries=5):
    retry_count = 0
    wait_time = 10 
    
    while retry_count < max_retries:
        try:
            audio_part = Part.from_data(data=audio_bytes, mime_type="audio/mpeg")
            response = model.generate_content([prompt_text, audio_part])
            return response.text.strip()
        
        except exceptions.ResourceExhausted:
            print(f"  [!] 触发限流 (429)，等待 {wait_time} 秒后重试 (第 {retry_count+1}/{max_retries} 次)...")
            time.sleep(wait_time)
            retry_count += 1
            wait_time *= 2 
            
        except exceptions.InternalServerError:
            print(f"  [!] 服务器内部错误 (500)，等待 5 秒重试...")
            time.sleep(5)
            retry_count += 1
            
        except Exception as e:
            raise e
            
    raise Exception("重试次数耗尽，API 调用失败")

# 4. 核心处理循环
processed_count = 0
skipped_count = 0

for zip_blob, video_files in task_list:
    subject_name = zip_blob.name.split('/')[-1].replace('.zip', '')
    
    subject_dir = os.path.join(RESULT_BASE_DIR, subject_name)
    os.makedirs(subject_dir, exist_ok=True)
    
    local_zip = os.path.join(LOCAL_BASE_DIR, "current.zip")
    print(f"\n>>> 正在下载并解压主题: {subject_name}")
    zip_blob.download_to_filename(local_zip)
    
    try:
        with zipfile.ZipFile(local_zip, 'r') as z:
            z.extractall(subject_dir)
    except zipfile.BadZipFile:
        print(f"错误: 压缩包损坏 {subject_name}，跳过")
        if os.path.exists(local_zip): os.remove(local_zip)
        continue
    
    for root, _, files in os.walk(subject_dir):
        for file in files:
            if file.lower().endswith(('.mp4', '.mov', '.avi')) and not file.startswith('._'):
                processed_count += 1
                video_path = os.path.join(root, file)
                audio_path = os.path.join(root, os.path.splitext(file)[0] + ".mp3")
                txt_name = f"{os.path.splitext(file)[0]}.txt"
                
                cloud_txt_path = f"{CLOUD_RESULT_PREFIX}/{subject_name}/{txt_name}"

                if cloud_txt_path in existing_blobs:
                    print(f"[{processed_count}/{total_videos}] 跳过已存在: {file}")
                    skipped_count += 1
                    if os.path.exists(video_path): os.remove(video_path)
                    continue
                
                percent = (processed_count / total_videos) * 100
                print(f"进度: [{percent:.2f}%] | 正在处理: {file}")
                
                try:
                    # 第一步：提取音频
                    subprocess.run([
                        'ffmpeg', '-i', video_path, 
                        '-vn', '-acodec', 'libmp3lame', '-q:a', '2', 
                        audio_path
                    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                    
                    # 第二步：调用 Gemini
                    prompt = """
                    逐字转录脱口秀原文，并在每个段子或笑点结束后强制换行分段。直接输出原文。
                    """
                    
                    with open(audio_path, "rb") as f_audio:
                        audio_data = f_audio.read()
                    
                    result_text = generate_with_retry(prompt, audio_data)
                    
                    # 第三步：生成 TXT
                    txt_local_path = os.path.join(subject_dir, txt_name)
                    os.makedirs(os.path.dirname(txt_local_path), exist_ok=True)
                    
                    with open(txt_local_path, "w", encoding="utf-8") as f_txt:
                        f_txt.write(result_text)
                    
                    # 上传云端
                    bucket.blob(cloud_txt_path).upload_from_filename(txt_local_path)
                    existing_blobs.add(cloud_txt_path)
                    
                    # 第四步：清理本地 视频和音频
                    if os.path.exists(video_path): os.remove(video_path)
                    if os.path.exists(audio_path): os.remove(audio_path)
                    
                except Exception as e:
                    print(f"处理文件 {file} 失败: {e}")
                    if os.path.exists(audio_path): os.remove(audio_path)

    if os.path.exists(local_zip):
        os.remove(local_zip)
    
    print(f"主题 {subject_name} 处理完毕，TXT文件已保留在本地。")

print(f"\n--- 任务全部完成！共扫描 {processed_count}，实际处理 {processed_count - skipped_count}，跳过 {skipped_count} ---")

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_supp

正在同步云端 [talkshow_collection_content] 下已完成的任务列表...
云端已存在 0 个解析结果，这些将被跳过。

正在预扫描视频总量以计算全局进度...
共计待处理视频总数: 144


>>> 正在下载并解压主题: 小红书博主【脱口秀集锦】笔记批量下载_20260210121117
进度: [0.69%] | 正在处理: 脱口秀 孟川 我真的太爱呆在厕所里了_1.mp4
进度: [1.39%] | 正在处理: 脱口秀 孟川 薄利多销_1.mp4
进度: [2.08%] | 正在处理: 脱口秀 陈艾 我不是精神病我只是先疯_1.mp4
进度: [2.78%] | 正在处理: 脱口秀 翟佳宁 那是太胖了吗_1.mp4
进度: [3.47%] | 正在处理: 脱口秀 王越 很空虚我没有底_1.mp4
进度: [4.17%] | 正在处理: 脱口秀 陈艾 师傅麻烦你我要大火_1.mp4
进度: [4.86%] | 正在处理: 蛛丝缠心_1.mp4
进度: [5.56%] | 正在处理: 脱口秀 付航 下一次有不喜欢吃的提前说_1.mp4
进度: [6.25%] | 正在处理: 脱口秀 孟川 短发光头和披肩发_1.mp4
进度: [6.94%] | 正在处理: 夜潮寂静_1.mp4
进度: [7.64%] | 正在处理: 脱口秀 孟川 量子纠缠_1.mp4
进度: [8.33%] | 正在处理: 脱口秀 张踩铃 你只是那个时候还不知道_1.mp4
进度: [9.03%] | 正在处理: 脱口秀 罗永浩 找回了他的创作灵感_1.mp4
进度: [9.72%] | 正在处理: 脱口秀 张踩铃 没用的人活得更爽_1.mp4
进度: [10.42%] | 正在处理: 脱口秀 赵越 一个月八百_1.mp4
进度: [11.11%] | 正在处理: 脱口秀 翟佳宁 熊出没_1.mp4
进度: [11.81%] | 正在处理: 脱口秀 良言 他这音儿疑似歧视外地人_1.mp4
进度: [12.50%] | 正在处理: 脱口秀 刘仁铖 可能是因为你没参加过高考_1.mp4
进度: [13.19%] | 正在处理: 脱口秀 漫才兄弟 扮演谈判专家_1.mp4
主题 小红书博主【脱口秀集锦】笔记批量下载_20260210121117 处理完毕，TXT文件已保留在